In [ ]:
import pandas as pd
import numpy as np

In [ ]:
products = pd.read_csv('/content/combined_products_2025_v2-2.csv')

In [ ]:
suppliers = pd.read_csv('/content/synthetic_suppliers.csv')

In [ ]:
products.loc[lambda x: x['product'] == 'power_devices']

,date,ppi_value,country_code,ppi_source,real_price,product,ppi_region,year,month
22700,2012-02-01,NaN,JPN,FRED,NaN,power_devices,OAC,2012,2
22701,2012-02-01,83.8000,DEU,FRED,0.125700,power_devices,USA,2012,2
22702,2012-02-01,83.8000,FRA,FRED,0.117320,power_devices,USA,2012,2
22703,2012-02-01,83.8000,GBR,FRED,0.117320,power_devices,USA,2012,2
22704,2012-02-01,83.8000,NLD,FRED,0.117320,power_devices,USA,2012,2
...,...,...,...,...,...,...,...,...,...
24365,2025-12-01,98.7000,SGP,FRED,0.088830,power_devices,OAC,2025,12
24366,2025-12-01,98.7000,MYS,FRED,0.078960,power_devices,OAC,2025,12
24367,2025-12-01,98.7000,THA,FRED,0.076986,power_devices,OAC,2025,12
24368,2025-12-01,88.2378,CHN,FRED,0.061766,power_devices,CHN,2025,12


In [ ]:
suppliers

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability
0,ARE,SUP_ARE_1,127.022009,9.676976,93.643856,0.245273,0.661656,0.825186
1,AUS,SUP_AUS_2,101.661492,7.052135,49.732605,0.212867,0.810191,0.759953
2,AUS,SUP_AUS_3,97.134227,6.717316,45.122336,0.199328,0.860594,0.793081
3,AUS,SUP_AUS_4,110.434919,7.071551,50.006838,0.199408,0.856211,0.774636
4,AUS,SUP_AUS_5,102.386666,7.089366,50.259116,0.213793,0.761370,0.803894
...,...,...,...,...,...,...,...,...
86,THA,SUP_THA_87,95.394628,5.507385,30.331291,0.454858,0.498049,0.712618
87,USA,SUP_USA_88,68.049751,5.456252,29.770688,0.342055,0.755714,0.751810
88,USA,SUP_USA_89,65.482029,4.526440,20.488657,0.290528,0.738044,0.764231
89,USA,SUP_USA_90,63.552058,5.599639,31.355957,0.342055,0.789330,0.787959


In [ ]:
suppliers.country_code.unique()

array(['ARE', 'AUS', 'BEL', 'BRA', 'CAN', 'CHN', 'DEU', 'FIN', 'FRA',
       'GBR', 'HKG', 'IDN', 'IND', 'JPN', 'MEX', 'MYS', 'NLD', 'SGP',
       'THA', 'USA'], dtype=object)

In [ ]:
products['product'].unique()

array(['integrated_circuit_components', 'microprocessors',
       'power_devices', 'transistors'], dtype=object)

# Attaching Products Randomly to Suppliers by Weighted-Tier Levels

In [ ]:
# ============================================================
# Country-Specific, Industry-Aligned Product Supply Weights
# ============================================================

country_product_weights = {

    # -------------------------
    # Tier 1A — IC Component & OSAT Powerhouses
    # -------------------------
    "CHN": {
        "Microprocessors": 0.10,
        "Power Devices": 0.20,
        "Transistors": 0.25,
        "IC Components": 0.45
    },
    "MYS": {
        "Microprocessors": 0.10,
        "Power Devices": 0.20,
        "Transistors": 0.20,
        "IC Components": 0.50
    },
    "THA": {
        "Microprocessors": 0.05,
        "Power Devices": 0.25,
        "Transistors": 0.20,
        "IC Components": 0.50
    },
    "SGP": {
        "Microprocessors": 0.15,
        "Power Devices": 0.20,
        "Transistors": 0.15,
        "IC Components": 0.50
    },

    # -------------------------
    # Tier 1B — Advanced Logic / Specialty Analog
    # -------------------------
    "USA": {
        "Microprocessors": 0.20,
        "Power Devices": 0.05,
        "Transistors": 0.70,
        "IC Components": 0.05
    },
    "JPN": {
        "Microprocessors": 0.20,
        "Power Devices": 0.40,
        "Transistors": 0.25,
        "IC Components": 0.15
    },
    "DEU": {
        "Microprocessors": 0.10,
        "Power Devices": 0.45,
        "Transistors": 0.25,
        "IC Components": 0.20
    },

    # -------------------------
    # Tier 2 — Moderate Semiconductor Presence
    # -------------------------
    "MEX": {
        "Microprocessors": 0.10,
        "Power Devices": 0.25,
        "Transistors": 0.30,
        "IC Components": 0.35
    },
    "NLD": {
        "Microprocessors": 0.20,
        "Power Devices": 0.25,
        "Transistors": 0.25,
        "IC Components": 0.30
    },
    "CAN": {
        "Microprocessors": 0.20,
        "Power Devices": 0.05,
        "Transistors": 0.35,
        "IC Components": 0.40
    },
    "FRA": {
        "Microprocessors": 0.15,
        "Power Devices": 0.30,
        "Transistors": 0.25,
        "IC Components": 0.30
    },
    "GBR": {
        "Microprocessors": 0.20,
        "Power Devices": 0.20,
        "Transistors": 0.30,
        "IC Components": 0.30
    },
    "IND": {
        "Microprocessors": 0.10,
        "Power Devices": 0.25,
        "Transistors": 0.30,
        "IC Components": 0.35
    },
    "HKG": {
        "Microprocessors": 0.10,
        "Power Devices": 0.05,
        "Transistors": 0.40,
        "IC Components": 0.45
    },
    "IDN": {
        "Microprocessors": 0.10,
        "Power Devices": 0.05,
        "Transistors": 0.40,
        "IC Components": 0.45
    },

    # -------------------------
    # Tier 3 — Low Semiconductor Strength
    # -------------------------
    "ARE": {
        "Microprocessors": 0.05,
        "Power Devices": 0.15,
        "Transistors": 0.55,
        "IC Components": 0.25
    },
    "AUS": {
        "Microprocessors": 0.10,
        "Power Devices": 0.05,
        "Transistors": 0.55,
        "IC Components": 0.30
    },
    "BEL": {
        "Microprocessors": 0.10,
        "Power Devices": 0.20,
        "Transistors": 0.40,
        "IC Components": 0.30
    },
    "BRA": {
        "Microprocessors": 0.05,
        "Power Devices": 0.20,
        "Transistors": 0.45,
        "IC Components": 0.30
    },
    "FIN": {
        "Microprocessors": 0.10,
        "Power Devices": 0.20,
        "Transistors": 0.40,
        "IC Components": 0.30
    }
}


In [ ]:
# ============================================================
# Function to sample a product for each supplier
# ============================================================

random_state = 42

def assign_product(country_code):
    weights = country_product_weights[country_code]
    products = list(weights.keys())
    probs = np.array(list(weights.values()))
    return np.random.choice(products, p=probs)

# ============================================================
# Apply to your suppliers DataFrame
# ============================================================

suppliers["product"] = suppliers["country_code"].apply(assign_product)


In [ ]:
suppliers.tail(30)

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product
61,MYS,SUP_MYS_62,93.674561,5.874361,34.508121,0.408347,0.611556,0.709132,Transistors
62,MYS,SUP_MYS_63,92.603276,6.587282,43.392282,0.368410,0.607717,0.738115,IC Components
63,MYS,SUP_MYS_64,96.626675,5.521518,30.487166,0.416490,0.616157,0.711582,IC Components
64,NLD,SUP_NLD_65,99.248126,8.955970,80.209394,0.199309,0.797345,0.843008,Transistors
65,NLD,SUP_NLD_66,87.501044,7.577642,57.420661,0.205113,0.825017,0.797405,IC Components
66,NLD,SUP_NLD_67,90.568945,8.214084,67.471181,0.225406,0.833884,0.787090,IC Components
67,NLD,SUP_NLD_68,96.742753,7.825076,61.231819,0.206893,0.880993,0.802450,Power Devices
68,SGP,SUP_SGP_69,103.816474,7.031540,49.442561,0.142956,0.861526,0.869948,IC Components
69,SGP,SUP_SGP_70,104.685820,5.770543,33.299169,0.129999,0.940477,0.847667,IC Components
70,SGP,SUP_SGP_71,94.535090,5.595775,31.312693,0.137044,0.904991,0.919955,Power Devices


# Creating a Probability of Defect column based on what the country behind the supplier *tends* to manufacture in the modern semiconductor industry

Criteria:
 - How aligned the supplier's assigned product is with what the country is actually known for producing
 - how mature and reliable that country's manufacturing base is
 - How complex the product category is (microprocessors are more difficult to manufacture; thus, higher defect risk)
 - Added a little noise for variation within each countries list of suppliers

In [ ]:
import numpy as np

# ------------------------------------------------------------
# Baseline difficulty per product
# ------------------------------------------------------------
product_difficulty = {
    "Microprocessors": 0.08,
    "Power Devices": 0.05,
    "Transistors": 0.03,
    "IC Components": 0.04
}

# ------------------------------------------------------------
# Function to compute probability_of_defect
# ------------------------------------------------------------

random_state = 42

def compute_defect_probability(country_code, product):
    # Get the country’s weight for this product
    weight = country_product_weights[country_code][product]

    # Alignment factor (higher = worse alignment)
    alignment_factor = 1 - weight

    # Baseline difficulty
    difficulty = product_difficulty[product]

    # Final defect probability
    defect_prob = difficulty * (0.5 + 0.5 * alignment_factor) * np.random.uniform(0.9, 1.1)

    return defect_prob


In [ ]:
# ------------------------------------------------------------
# Apply to suppliers DataFrame
# ------------------------------------------------------------
suppliers["probability_of_defect"] = suppliers.apply(
    lambda row: compute_defect_probability(row["country_code"], row["product"]),
    axis=1
)


In [ ]:
suppliers.tail(30)

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect
61,MYS,SUP_MYS_62,93.674561,5.874361,34.508121,0.408347,0.611556,0.709132,Transistors,0.028008
62,MYS,SUP_MYS_63,92.603276,6.587282,43.392282,0.368410,0.607717,0.738115,IC Components,0.027441
63,MYS,SUP_MYS_64,96.626675,5.521518,30.487166,0.416490,0.616157,0.711582,IC Components,0.030409
64,NLD,SUP_NLD_65,99.248126,8.955970,80.209394,0.199309,0.797345,0.843008,Transistors,0.026331
65,NLD,SUP_NLD_66,87.501044,7.577642,57.420661,0.205113,0.825017,0.797405,IC Components,0.035020
66,NLD,SUP_NLD_67,90.568945,8.214084,67.471181,0.225406,0.833884,0.787090,IC Components,0.036066
67,NLD,SUP_NLD_68,96.742753,7.825076,61.231819,0.206893,0.880993,0.802450,Power Devices,0.039596
68,SGP,SUP_SGP_69,103.816474,7.031540,49.442561,0.142956,0.861526,0.869948,IC Components,0.027652
69,SGP,SUP_SGP_70,104.685820,5.770543,33.299169,0.129999,0.940477,0.847667,IC Components,0.028997
70,SGP,SUP_SGP_71,94.535090,5.595775,31.312693,0.137044,0.904991,0.919955,Power Devices,0.043702


# Adding Bulk-Order Discount Rate Column

Criteria:
- how mature and high-volume a country's semiconductor manufacturing base is
- how aligned the supplier's assigned product is with what that country is actually known for producing
- how competitive the country is in packaging markets
- how aggressively suppliers in that country typically discount for volume

In [ ]:
random_state = 42

# ------------------------------------------------------------
# Base bulk discount by product (industry realistic)
# ------------------------------------------------------------
base_bulk_discount = {
    "Microprocessors": 0.05,   # 5%
    "Power Devices": 0.08,     # 8%
    "Transistors": 0.12,       # 12%
    "IC Components": 0.15      # 15%
}

# ------------------------------------------------------------
# Compute bulk discount based on country-product alignment
# ------------------------------------------------------------
def compute_bulk_discount(country_code, product):
    # Country's strength in this product
    weight = country_product_weights[country_code][product]

    # Base discount for this product category
    base = base_bulk_discount[product]

    # Final discount: stronger alignment → deeper discount
    discount = base * (0.5 + 0.5 * weight) * np.random.uniform(0.9, 1.1)

    return discount

# ------------------------------------------------------------
# Apply to suppliers DataFrame
# ------------------------------------------------------------
suppliers["bulk_discount"] = suppliers.apply(
    lambda row: compute_bulk_discount(row["country_code"], row["product"]),
    axis=1
)


In [ ]:
suppliers

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount
0,ARE,SUP_ARE_1,127.022009,9.676976,93.643856,0.245273,0.661656,0.825186,Power Devices,0.047943,0.044913
1,AUS,SUP_AUS_2,101.661492,7.052135,49.732605,0.212867,0.810191,0.759953,IC Components,0.034792,0.096975
2,AUS,SUP_AUS_3,97.134227,6.717316,45.122336,0.199328,0.860594,0.793081,Transistors,0.021262,0.091715
3,AUS,SUP_AUS_4,110.434919,7.071551,50.006838,0.199408,0.856211,0.774636,IC Components,0.031539,0.089374
4,AUS,SUP_AUS_5,102.386666,7.089366,50.259116,0.213793,0.761370,0.803894,Transistors,0.020360,0.100759
...,...,...,...,...,...,...,...,...,...,...,...
86,THA,SUP_THA_87,95.394628,5.507385,30.331291,0.454858,0.498049,0.712618,IC Components,0.028670,0.111955
87,USA,SUP_USA_88,68.049751,5.456252,29.770688,0.342055,0.755714,0.751810,Transistors,0.020456,0.097184
88,USA,SUP_USA_89,65.482029,4.526440,20.488657,0.290528,0.738044,0.764231,Transistors,0.020114,0.092853
89,USA,SUP_USA_90,63.552058,5.599639,31.355957,0.342055,0.789330,0.787959,Transistors,0.018405,0.104771


# Adding a bulk-units column i.e., bulk discounts only applied when tied to a minimum order quantity.

Depends on:
 - product category
 - country's manufacturing maturity
 - country's cost structure
 - country's alignment with that product (weight matrix)

Country alignment criteria:
 - If a country is strong in a product, MOQ is lower.
 - If weak, MOQ is higher.

In [ ]:
# ------------------------------------------------------------
# Base MOQ by product (industry realistic)
# ------------------------------------------------------------
random_state = 42

base_moq = {
    "Microprocessors": 1000,
    "Power Devices": 2000,
    "Transistors": 5000,
    "IC Components": 8000
}

# ------------------------------------------------------------
# Compute MOQ (bulk_units) based on country-product alignment
# ------------------------------------------------------------
def compute_bulk_units(country_code, product):
    weight = country_product_weights[country_code][product]
    base = base_moq[product]

    # Lower weight → higher MOQ; higher weight → lower MOQ
    multiplier = 1.2 - weight   # ranges ~0.7 to ~1.15

    moq = int(base * multiplier) * np.random.uniform(0.9, 1.1)

    # Ensure MOQ is at least 1
    return max(moq, 1)

# ------------------------------------------------------------
# Apply to suppliers DataFrame
# ------------------------------------------------------------
suppliers["bulk_units"] = suppliers.apply(
    lambda row: compute_bulk_units(row["country_code"], row["product"]),
    axis=1
)


In [ ]:
suppliers

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount,bulk_units
0,ARE,SUP_ARE_1,127.022009,9.676976,93.643856,0.245273,0.661656,0.825186,Power Devices,0.047943,0.044913,2233.900461
1,AUS,SUP_AUS_2,101.661492,7.052135,49.732605,0.212867,0.810191,0.759953,IC Components,0.034792,0.096975,7802.139449
2,AUS,SUP_AUS_3,97.134227,6.717316,45.122336,0.199328,0.860594,0.793081,Transistors,0.021262,0.091715,3301.429810
3,AUS,SUP_AUS_4,110.434919,7.071551,50.006838,0.199408,0.856211,0.774636,IC Components,0.031539,0.089374,7070.440209
4,AUS,SUP_AUS_5,102.386666,7.089366,50.259116,0.213793,0.761370,0.803894,Transistors,0.020360,0.100759,3413.244255
...,...,...,...,...,...,...,...,...,...,...,...,...
86,THA,SUP_THA_87,95.394628,5.507385,30.331291,0.454858,0.498049,0.712618,IC Components,0.028670,0.111955,5492.279921
87,USA,SUP_USA_88,68.049751,5.456252,29.770688,0.342055,0.755714,0.751810,Transistors,0.020456,0.097184,2706.954217
88,USA,SUP_USA_89,65.482029,4.526440,20.488657,0.290528,0.738044,0.764231,Transistors,0.020114,0.092853,2311.925787
89,USA,SUP_USA_90,63.552058,5.599639,31.355957,0.342055,0.789330,0.787959,Transistors,0.018405,0.104771,2475.604812


In [ ]:
products

,date,ppi_value,country_code,ppi_source,real_price,product,ppi_region,year,month
0,1976-01-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,1
1,1976-02-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,2
2,1976-03-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,3
3,1976-04-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,4
4,1976-05-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,5
...,...,...,...,...,...,...,...,...,...
36125,2025-12-01,98.7,BRA,FRED,0.041454,transistors,OAC,2025,12
36126,2025-12-01,98.7,IND,FRED,0.031584,transistors,OAC,2025,12
36127,2025-12-01,98.7,THA,FRED,0.034545,transistors,OAC,2025,12
36128,2025-12-01,98.7,IDN,FRED,0.032571,transistors,OAC,2025,12


# Creating a baseline price column

In [ ]:
product_name_map = {
    "IC Components": "integrated_circuit_components",
    "Transistors": "transistors",
    "Power Devices": "power_devices",
    "Microprocessors": "microprocessors"
}


In [ ]:
products.loc[lambda x: x['product'] == 'transistors']

,date,ppi_value,country_code,ppi_source,real_price,product,ppi_region,year,month
24370,1977-01-01,109.0,USA,BLS,0.070964,transistors,USA,1977,1
24371,1977-01-01,NaN,JPN,FRED,NaN,transistors,OAC,1977,1
24372,1977-01-01,NaN,DEU,FRED,NaN,transistors,USA,1977,1
24373,1977-01-01,NaN,FRA,FRED,NaN,transistors,USA,1977,1
24374,1977-01-01,NaN,GBR,FRED,NaN,transistors,USA,1977,1
...,...,...,...,...,...,...,...,...,...
36125,2025-12-01,98.7,BRA,FRED,0.041454,transistors,OAC,2025,12
36126,2025-12-01,98.7,IND,FRED,0.031584,transistors,OAC,2025,12
36127,2025-12-01,98.7,THA,FRED,0.034545,transistors,OAC,2025,12
36128,2025-12-01,98.7,IDN,FRED,0.032571,transistors,OAC,2025,12


In [ ]:
# Make a copy of products with normalized product names
products_copy = products.copy()
products_copy["product_norm"] = products_copy["product"].str.lower().str.strip()

# Compute most recent real_price per (country_code, product)
latest_prices = (
    products_copy.sort_values("date")
    .groupby(["country_code", "product_norm"])["real_price"]
    .last()
    .reset_index()
)


In [ ]:
# Normalize supplier product names to match products table
suppliers["product_norm"] = suppliers["product"].map(product_name_map)

# Merge baseline prices
suppliers = suppliers.merge(
    latest_prices,
    how="left",
    left_on=["country_code", "product_norm"],
    right_on=["country_code", "product_norm"]
)

# Rename for clarity
suppliers = suppliers.rename(columns={"real_price": "baseline_price"})


In [ ]:
suppliers = suppliers.drop(columns=["product_norm"])


In [ ]:
suppliers.drop(suppliers.loc[lambda x: x['baseline_price'].isna()].index, axis=0, inplace=True)

In [ ]:
suppliers.loc[lambda x: x['country_code']== 'THA']

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount,bulk_units,baseline_price
78,THA,SUP_THA_79,100.277387,5.412812,29.298529,0.489983,0.477038,0.710978,IC Components,0.027528,0.107124,5959.783326,0.140859
79,THA,SUP_THA_80,98.681799,6.505389,42.320083,0.475889,0.445450,0.690970,Power Devices,0.045207,0.048629,1743.459880,0.076986
80,THA,SUP_THA_81,100.475495,5.644594,31.861442,0.446817,0.487221,0.682779,Power Devices,0.047444,0.047585,1958.590102,0.076986
81,THA,SUP_THA_82,96.224869,6.099137,37.199468,0.467973,0.483340,0.686891,Transistors,0.028166,0.064884,4859.780925,0.034545
82,THA,SUP_THA_83,98.083843,5.424881,29.429330,0.498967,0.518076,0.725676,Power Devices,0.046885,0.046392,2079.209177,0.076986
83,THA,SUP_THA_84,96.502208,5.437766,29.569298,0.480340,0.485508,0.704155,Power Devices,0.041780,0.053941,1901.937853,0.076986
84,THA,SUP_THA_85,95.701557,6.085417,37.032304,0.475536,0.478193,0.702526,IC Components,0.027163,0.122454,5814.506067,0.140859
85,THA,SUP_THA_86,98.579174,5.754680,33.116337,0.430113,0.518949,0.696473,IC Components,0.032911,0.109232,5716.613567,0.140859
86,THA,SUP_THA_87,95.394628,5.507385,30.331291,0.454858,0.498049,0.712618,IC Components,0.028670,0.111955,5492.279921,0.140859


In [ ]:
suppliers = suppliers.assign(
    bulk_units = lambda x: x.bulk_units.astype(int)
)

In [ ]:
suppliers.tail()

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount,bulk_units,baseline_price
86,THA,SUP_THA_87,95.394628,5.507385,30.331291,0.454858,0.498049,0.712618,IC Components,0.028670,0.111955,5492,0.140859
87,USA,SUP_USA_88,68.049751,5.456252,29.770688,0.342055,0.755714,0.751810,Transistors,0.020456,0.097184,2706,0.035878
88,USA,SUP_USA_89,65.482029,4.526440,20.488657,0.290528,0.738044,0.764231,Transistors,0.020114,0.092853,2311,0.035878
89,USA,SUP_USA_90,63.552058,5.599639,31.355957,0.342055,0.789330,0.787959,Transistors,0.018405,0.104771,2475,0.035878
90,USA,SUP_USA_91,66.503117,5.185243,26.886741,0.344072,0.729554,0.803135,Transistors,0.020099,0.111044,2371,0.035878


# Injecting Some Noise into Baseline Prices by Supplier

In [ ]:
random_state = 42

# Product-specific noise ranges (multiplicative)
product_noise_ranges = {
    "Microprocessors": 0.03,   # ±3%
    "Power Devices": 0.05,     # ±5%
    "Transistors": 0.08,       # ±8%
    "IC Components": 0.10      # ±10%
}

def add_product_specific_noise(row):
    base_price = row["baseline_price"]
    product = row["product"]

    # Get noise scale for this product
    noise_scale = product_noise_ranges[product]

    # Generate multiplicative noise factor
    noise_factor = np.random.uniform(1 - noise_scale, 1 + noise_scale)

    return base_price * noise_factor




In [ ]:
# Apply to suppliers DataFrame
suppliers["baseline_price"] = suppliers.apply(add_product_specific_noise, axis=1)

In [ ]:
suppliers.tail()

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount,bulk_units,baseline_price
86,THA,SUP_THA_87,95.394628,5.507385,30.331291,0.454858,0.498049,0.712618,IC Components,0.028670,0.111955,5492,0.148515
87,USA,SUP_USA_88,68.049751,5.456252,29.770688,0.342055,0.755714,0.751810,Transistors,0.020456,0.097184,2706,0.033626
88,USA,SUP_USA_89,65.482029,4.526440,20.488657,0.290528,0.738044,0.764231,Transistors,0.020114,0.092853,2311,0.038409
89,USA,SUP_USA_90,63.552058,5.599639,31.355957,0.342055,0.789330,0.787959,Transistors,0.018405,0.104771,2475,0.034622
90,USA,SUP_USA_91,66.503117,5.185243,26.886741,0.344072,0.729554,0.803135,Transistors,0.020099,0.111044,2371,0.033298


In [ ]:
products

,date,ppi_value,country_code,ppi_source,real_price,product,ppi_region,year,month
0,1976-01-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,1
1,1976-02-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,2
2,1976-03-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,3
3,1976-04-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,4
4,1976-05-01,100.0,ARE,FRED,0.260000,integrated_circuit_components,OAC,1976,5
...,...,...,...,...,...,...,...,...,...
36125,2025-12-01,98.7,BRA,FRED,0.041454,transistors,OAC,2025,12
36126,2025-12-01,98.7,IND,FRED,0.031584,transistors,OAC,2025,12
36127,2025-12-01,98.7,THA,FRED,0.034545,transistors,OAC,2025,12
36128,2025-12-01,98.7,IDN,FRED,0.032571,transistors,OAC,2025,12


# Creating a Price Volatility for each supplier based on their disruption probability and rolling standard deviation of real_price of product (grouped by country in products table) over last 5 years

In [ ]:
product_map = {
    "IC Components": "integrated_circuit_components",
    "Transistors": "transistors",
    "Power Devices": "power_devices",
    "Microprocessors": "microprocessors"
}

suppliers["product_norm"] = suppliers["product"].map(product_map)
products["product_norm"] = products["product"].str.lower().str.strip()


In [ ]:
products_sorted = products.sort_values(["country_code", "product_norm", "date"])

products_sorted["price_volatility_raw"] = (
    products_sorted
    .groupby(["country_code", "product_norm"])["real_price"]
    .rolling(60, min_periods=12)
    .std()
    .reset_index(level=[0,1], drop=True)
)


In [ ]:
latest_volatility = (
    products_sorted
    .groupby(["country_code", "product_norm"])["price_volatility_raw"]
    .last()
    .reset_index()
)


In [ ]:
suppliers = suppliers.merge(
    latest_volatility,
    how="left",
    left_on=["country_code", "product_norm"],
    right_on=["country_code", "product_norm"]
)


In [ ]:
min_v = suppliers["price_volatility_raw"].min()
max_v = suppliers["price_volatility_raw"].max()

suppliers["price_volatility_norm"] = (
    (suppliers["price_volatility_raw"] - min_v) / (max_v - min_v)
)


In [ ]:
suppliers["price_volatility"] = (
    0.6 * suppliers["price_volatility_norm"] +
    0.4 * suppliers["disruption_probability"]
)


In [ ]:
suppliers = suppliers.drop(columns=["price_volatility_raw", "price_volatility_norm",
                                    'product_norm'])


In [ ]:
suppliers

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount,bulk_units,baseline_price,price_volatility
0,AUS,SUP_AUS_2,101.661492,7.052135,49.732605,0.212867,0.810191,0.759953,IC Components,0.034792,0.096975,7802,0.207229,0.131251
1,AUS,SUP_AUS_3,97.134227,6.717316,45.122336,0.199328,0.860594,0.793081,Transistors,0.021262,0.091715,3301,0.049958,0.087788
2,AUS,SUP_AUS_4,110.434919,7.071551,50.006838,0.199408,0.856211,0.774636,IC Components,0.031539,0.089374,7070,0.194614,0.125868
3,AUS,SUP_AUS_5,102.386666,7.089366,50.259116,0.213793,0.761370,0.803894,Transistors,0.020360,0.100759,3413,0.046048,0.093574
4,BEL,SUP_BEL_6,97.521285,11.052010,122.146934,0.256284,0.823733,0.794199,Microprocessors,0.083262,0.025069,1009,0.785374,0.544780
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,THA,SUP_THA_87,95.394628,5.507385,30.331291,0.454858,0.498049,0.712618,IC Components,0.028670,0.111955,5492,0.148515,0.212679
85,USA,SUP_USA_88,68.049751,5.456252,29.770688,0.342055,0.755714,0.751810,Transistors,0.020456,0.097184,2706,0.033626,0.140960
86,USA,SUP_USA_89,65.482029,4.526440,20.488657,0.290528,0.738044,0.764231,Transistors,0.020114,0.092853,2311,0.038409,0.120349
87,USA,SUP_USA_90,63.552058,5.599639,31.355957,0.342055,0.789330,0.787959,Transistors,0.018405,0.104771,2475,0.034622,0.140960


In [ ]:
suppliers.loc[lambda x: x['product'] == 'Transistors']

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount,bulk_units,baseline_price,price_volatility,hts8_tariff
1,AUS,SUP_AUS_3,97.134227,6.717316,45.122336,0.199328,0.860594,0.793081,Transistors,0.021262,0.091715,3301,0.049958,0.087788,None
3,AUS,SUP_AUS_5,102.386666,7.089366,50.259116,0.213793,0.761370,0.803894,Transistors,0.020360,0.100759,3413,0.046048,0.093574,None
5,BEL,SUP_BEL_7,104.651825,11.391627,129.769171,0.266364,0.815507,0.838855,Transistors,0.022027,0.078711,4002,0.045940,0.108508,None
6,BRA,SUP_BRA_8,103.812191,10.363937,107.411192,0.462936,0.472465,0.681343,Transistors,0.023058,0.094425,3794,0.043447,0.192225,None
8,CAN,SUP_CAN_10,71.551361,7.364183,54.231184,0.217086,0.885767,0.803172,Transistors,0.025323,0.074381,4051,0.045657,0.087346,None
9,CAN,SUP_CAN_11,79.421827,8.019880,64.318481,0.227178,0.805211,0.810051,Transistors,0.026068,0.084170,4670,0.041250,0.091382,None
13,CHN,SUP_CHN_15,95.289012,5.551037,30.814013,0.459787,0.467951,0.773362,Transistors,0.025850,0.078837,4753,0.024819,0.184029,None
15,CHN,SUP_CHN_17,94.110556,6.031509,36.379098,0.397835,0.500291,0.760977,Transistors,0.026535,0.081741,4500,0.025937,0.159248,None
17,DEU,SUP_DEU_19,89.065428,5.157890,26.603825,0.246849,0.826595,0.791051,Transistors,0.027913,0.076225,5116,0.040417,0.100702,None
18,DEU,SUP_DEU_20,84.563994,4.295237,18.449057,0.211597,0.823429,0.812363,Transistors,0.027734,0.072157,4947,0.041633,0.086601,None


# Considering How to Add Tariff Info

In [ ]:
tariff = pd.read_csv('/content/tarriff_database_2025_only_semiconductor_components_v2.csv')

In [ ]:
tariff.loc[lambda x: x.mfn_text_rate_pct > 0]

,hts8,brief_description,mfn_text_rate_pct,how_measured
10,84807180,"Molds for rubber or plastics, injection or com...",3.1,NO
14,84879000,"Machinery parts, not containing electrical con...",3.9,KG
17,85042300,Liquid dielectric transformers having a power ...,1.6,NO
19,85043140,Electrical transformers other than liquid diel...,6.6,NO
20,85043160,Electrical transformers other than liquid diel...,1.6,NO
21,85043200,Electrical transformers other than liquid diel...,2.4,NO
22,85043300,Electrical transformers other than liquid diel...,1.6,NO
23,85043400,Electrical transformers other than liquid diel...,1.6,NO
31,85118020,Voltage and voltage-current regulators with cu...,2.5,NO
33,85119020,Parts of voltage and voltage-current regulator...,3.1,KG


Code 85389030 can potentially be included under IC components

In [ ]:
suppliers = suppliers.assign(
    hts8_tariff = lambda x: np.where(x['product'] == 'IC Components', '85389030', 'None')
)

In [ ]:
suppliers.to_csv('suppliers_products.csv', index=False)

In [ ]:
suppliers.head(30)

,country_code,supplier_id,lead_time_mean,lead_time_stddev,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount,bulk_units,baseline_price,price_volatility,hts8_tariff
0,AUS,SUP_AUS_2,101.661492,7.052135,49.732605,0.212867,0.810191,0.759953,IC Components,0.034792,0.096975,7802,0.207229,0.131251,85389030
1,AUS,SUP_AUS_3,97.134227,6.717316,45.122336,0.199328,0.860594,0.793081,Transistors,0.021262,0.091715,3301,0.049958,0.087788,None
2,AUS,SUP_AUS_4,110.434919,7.071551,50.006838,0.199408,0.856211,0.774636,IC Components,0.031539,0.089374,7070,0.194614,0.125868,85389030
3,AUS,SUP_AUS_5,102.386666,7.089366,50.259116,0.213793,0.761370,0.803894,Transistors,0.020360,0.100759,3413,0.046048,0.093574,None
4,BEL,SUP_BEL_6,97.521285,11.052010,122.146934,0.256284,0.823733,0.794199,Microprocessors,0.083262,0.025069,1009,0.785374,0.544780,None
5,BEL,SUP_BEL_7,104.651825,11.391627,129.769171,0.266364,0.815507,0.838855,Transistors,0.022027,0.078711,4002,0.045940,0.108508,None
6,BRA,SUP_BRA_8,103.812191,10.363937,107.411192,0.462936,0.472465,0.681343,Transistors,0.023058,0.094425,3794,0.043447,0.192225,None
7,CAN,SUP_CAN_9,72.449212,8.063617,65.021927,0.194035,0.790818,0.871330,IC Components,0.031400,0.107729,5777,0.236929,0.077743,85389030
8,CAN,SUP_CAN_10,71.551361,7.364183,54.231184,0.217086,0.885767,0.803172,Transistors,0.025323,0.074381,4051,0.045657,0.087346,None
9,CAN,SUP_CAN_11,79.421827,8.019880,64.318481,0.227178,0.805211,0.810051,Transistors,0.026068,0.084170,4670,0.041250,0.091382,None
